In [ ]:
# !pip install torch_geometric

In [ ]:
# import all libraries needed downstream
import os
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

In [ ]:
# !pip install torch_scatter

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
system_size = 118

In [ ]:
# Load the data
data = np.load(f'/home/oarowolo/workfile/OPFData/{system_size}bus_nminusone_combined_dataset.npz',allow_pickle=True)

# Get all keys
print("Available keys in the dataset:")
for key in data.files:
    # Print the key and its array shape
    print(f"{key}: shape {data[key].shape}")

In [ ]:
grid_bus = data['grid_bus']  
grid_generator = data['grid_generator']
grid_load = data['grid_load']
grid_shunt = data['grid_shunt']
grid_ac_line_features = data['grid_ac_line_features']
grid_transformer_features = data['grid_transformer_features']
grid_ac_line_receivers = data['grid_ac_line_receivers']
grid_ac_line_senders = data['grid_ac_line_senders']
grid_transformer_senders = data['grid_transformer_senders']
grid_transformer_receivers = data['grid_transformer_receivers']
grid_generator_link_receivers = data['grid_generator_link_senders']
solution_bus = data['solution_bus']  
solution_generator = data['solution_generator'] 
solution_objective = data['metadata_objective']
solution_objective = solution_objective.reshape(-1,1)
grid_generator_link_receivers = data['grid_generator_link_receivers']
grid_load_link_receivers = data['grid_load_link_receivers']
grid_shunt_link_receivers = data['grid_shunt_link_receivers']

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles are a NumPy array
    angles = np.asarray(angles)
    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
node_inputs = []

master_branch_list = []

edge_inputs = []

node_outputs = []

objectives = []

gen_indices = []


In [ ]:
for k in range(grid_bus.shape[0]):

    node_input_instance = np.zeros((system_size,19))
    

    generator_indices = grid_generator_link_receivers[k].astype(np.int32)
    load_indices = grid_load_link_receivers[k].astype(np.int32)
    shunt_indices = grid_shunt_link_receivers[k].astype(np.int32)
    node_input_instance[:,:4] = grid_bus[k]
    node_input_instance[generator_indices,4:15] = grid_generator[k]
    node_input_instance[load_indices,15:17] = grid_load[k]
    node_input_instance[shunt_indices,17:19] = grid_shunt[k]
    node_inputs.append(node_input_instance) 

    gen_indices.append(generator_indices)

    branch_list = list(zip(grid_ac_line_senders[k], grid_ac_line_receivers[k]))
    transformer_list = list(zip(grid_transformer_senders[k], grid_transformer_receivers[k]))
    for j in transformer_list:
        branch_list.append(j)
    master_branch_list.append(branch_list)
    edge_input_instance = np.zeros((len(branch_list),11))
    edge_input_instance[:grid_ac_line_features[k].shape[0],:9] = grid_ac_line_features[k]
    edge_input_instance[grid_ac_line_features[k].shape[0]:,:2] =  grid_transformer_features[k][:,:2]
    edge_input_instance[grid_ac_line_features[k].shape[0]:,2:4] =  grid_transformer_features[k][:,9:]
    edge_input_instance[grid_ac_line_features[k].shape[0]:,4:9] =  grid_transformer_features[k][:,2:7]
    edge_input_instance[grid_ac_line_features[k].shape[0]:,9:] =  grid_transformer_features[k][:,7:9]
    edge_input_instance[:grid_ac_line_features[k].shape[0],9:10] = 1.0
    edge_inputs.append(edge_input_instance)

    node_output_instance = np.zeros((system_size,4))

    node_output_instance[:,:2] = solution_bus[k].astype(np.float32)
    node_output_instance[generator_indices,2:] = solution_generator[k].astype(np.float32)

    node_outputs.append(node_output_instance)

    objectives.append(solution_objective[k].astype(np.float32))


In [ ]:
def train_val_test_split(solution_bus, train_ratio=0.9, val_ratio=0.05, test_ratio=0.05, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = solution_bus.shape[0]
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    return train_indices,val_indices,test_indices

In [ ]:
train_indices, val_indices, test_indices = train_val_test_split(solution_bus)

In [ ]:
# function to create graph dataset including edge attributes
def create_graph_data(node_input, node_output, solution_obj, edge_master_list, edge_features,data_indices,gen_indices):

    dataset = []


    for i in (data_indices):
        
        edge_list = edge_master_list[i]
        edge_list = [[edge[0], edge[1]] for edge in edge_list]
        edge_list = torch.tensor(np.array(edge_list).T,dtype=torch.int64)
        edge_attr = edge_features[i].reshape(len(edge_features[i]),-1)
        edge_attr = torch.tensor(edge_attr).float()
        graph_input = torch.tensor(node_input[i]).float()
        graph_output = torch.tensor(node_output[i]).float()
        graph_obj = torch.tensor(solution_obj[i]).float()
        number_of_nodes = graph_input.shape[0]
        generator_mask = torch.zeros(system_size,dtype=torch.float32)
        generator_mask[gen_indices[i].flatten()] = 1.0
        generator_mask = [generator_mask]
        

        
        dataset.append(Data(x=graph_input, y=graph_output, edge_index=edge_list,edge_attr =edge_attr, objective = graph_obj, num_nodes = number_of_nodes,gen_mask = generator_mask))

    return dataset

In [ ]:
train_dataset = create_graph_data(node_inputs, node_outputs, objectives, master_branch_list, edge_inputs,train_indices,gen_indices)

In [ ]:
val_dataset = create_graph_data(node_inputs, node_outputs, objectives, master_branch_list, edge_inputs,val_indices,gen_indices)

In [ ]:
test_dataset = create_graph_data(node_inputs, node_outputs, objectives, master_branch_list, edge_inputs,test_indices,gen_indices)

In [ ]:
batch_size = 256

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
## wandb set-up
api_key = '' ## insert your own api key
wandb.login(key=api_key)

In [ ]:
class SimpleGCN(torch.nn.Module):
    def __init__(
        self,
        hidden_size=256,
        n_mp_layers=5, # number of GNN layers, use graph diameter for example
        nfeature_dim=19,
        efeature_dim=11, # dimension of edge attributes
        output_dim = 4

    ):
        super().__init__()
        
        self.convs = nn.ModuleList()
        for i in range(n_mp_layers):
            in_dim = nfeature_dim if i == 0 else hidden_size
            out_dim = output_dim if i == n_mp_layers - 1 else hidden_size
            self.convs.append(GCNConv(in_dim, out_dim))
        
        self.relu = nn.ReLU()
            

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i != len(self.convs) - 1:
                x = self.relu(x)
        
        return torch.sigmoid(x)

In [ ]:
def prepare_batched_masks(batch_input, generator_masks, device):

    total_nodes = batch_input.shape[0]
    generator_mask = generator_masks[0]  # Since it's a list with single tensor
    
    # Create feature mask
    feature_mask = torch.zeros(total_nodes, 4, device=device)
    
    # Reshape generator mask to match expected output shape
    generator_mask = generator_mask.reshape(-1, 1)
    
    # Set feature mask for generator nodes (all 4 features)
    feature_mask[generator_mask.squeeze() == 1] = 1.0
    
    # Set feature mask for non-generator nodes (only first 2 features)
    feature_mask[generator_mask.squeeze() == 0, :2] = 1.0
    
    return generator_mask, feature_mask

In [ ]:
def masked_mse_loss(pred, target, feature_mask):

    # Apply mask to both predictions and targets
    masked_pred = pred * feature_mask
    masked_target = target * feature_mask
    
    # Calculate MSE loss
    loss = F.mse_loss(masked_pred, masked_target)
    
    return loss

In [ ]:
def convert_bounds(model_input,model_output):

    num_nodes = model_input.shape[0]

    out_size = model_output.shape[-1]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    pmin = model_input[:,6:7]

    pmax = model_input[:,7:8]

    
    qmin = model_input[:,9:10]

    qmax = model_input[:,10:11]

    thetamin =  torch.tensor([-1.0]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([1.0]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax, pmax, qmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin, pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    
#     for batch in tqdm(trainloader, desc="Training"):
    for batch in trainloader:
    
        optimizer.zero_grad()
        
        batch = batch.to(device)
        # Get batch size from the batch tensor
        batch_size = batch.batch.max().item() + 1
        
    
        # Prepare masks for the entire batch
        generator_mask, feature_mask = prepare_batched_masks(
            batch.x,
            batch.gen_mask,
            device=device
        )

        generator_mask = generator_mask.to(device)

        feature_mask = feature_mask.to(device)

        pred = model(batch)

        b_up, b_down = convert_bounds(batch.x,pred)
        b_up = b_up.to(device)
        b_down = b_down.to(device)
        pred = pred * (b_up - b_down) + b_down

        # Compute simple masked MSE loss
        loss = masked_mse_loss(pred, batch.y, feature_mask)
        masked_pred = pred * feature_mask

        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    
    return total_loss / len(trainloader)

In [ ]:
@torch.no_grad()
def validate_model(model, val_loader):

    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    
    for batch in val_loader:
#     for batch in tqdm(val_loader, desc=f"evaluating"):
        # Move batch to model's device
        batch = batch.to(device)
        
        # Get batch size from the batch tensor
        batch_size = batch.batch.max().item() + 1
        
        # Prepare masks for the entire batch
        generator_mask, feature_mask = prepare_batched_masks(
            batch.x,
            batch.gen_mask,
            device=device
        )

        generator_mask = generator_mask.to(device)

        feature_mask = feature_mask.to(device)
        
        pred = model(batch)

        b_up, b_down = convert_bounds(batch.x,pred)
        b_up = b_up.to(device)
        b_down = b_down.to(device)
        pred = pred * (b_up - b_down) + b_down
        
        masked_pred = pred * feature_mask
        
        # # Compute loss
        loss = masked_mse_loss(pred, batch.y, feature_mask)
        
        total_loss += loss.item() 
    
    return total_loss/len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    
    total_loss = 0.0
    all_predictions = []
    all_targets = []
    theta_predictions = []
    theta_targets = []
    
    for batch in testloader:

        batch = batch.to(device)
        
        # Get batch size from the batch tensor
        batch_size = batch.batch.max().item() + 1
        
        # Prepare masks for the entire batch
        generator_mask, feature_mask = prepare_batched_masks(
            batch.x,
            batch.gen_mask,
            device=device
        )
        
        # Forward pass

        generator_mask = generator_mask.to(device)

        feature_mask = feature_mask.to(device)
        
        pred = model(batch)

        b_up, b_down = convert_bounds(batch.x,pred)
        b_up = b_up.to(device)
        b_down = b_down.to(device)
        pred = pred * (b_up - b_down) + b_down

        masked_pred = pred * feature_mask
        masked_target = batch.y * feature_mask
        
        # Store predictions and targets for overall metrics
        all_predictions.append(masked_pred.cpu())
        all_targets.append(masked_target.cpu())
        
        # Compute loss
        loss = masked_mse_loss(pred, batch.y, feature_mask)

        total_loss += loss.item()

    
    return total_loss / len(testloader), all_predictions, all_targets

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"{system_size}_N-1_GCN_5_256_PQVT",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GCN",
      "dataset": "Full",
      "epochs": 100,
      })

In [ ]:
model = SimpleGCN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5,weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20) 

In [ ]:
model.load_state_dict(torch.load(f"/home/oarowolo/workfile/OPFData/{system_size}_bus_GCN_5_256_PQVT.pth"))

In [ ]:
model.eval()

In [ ]:
# training_losses = []
# validation_losses = []
# best_valid_loss = float('inf')
# early_stop_thresh = 100
# best_epoch = -1
# num_epochs = 100
# for epoch in tqdm(range(num_epochs), desc="Training Progress"):
#     train_loss = train_model(model, train_loader, optimizer)
#     valid_loss = validate_model(model, val_loader)
#     training_losses.append(train_loss)
#     validation_losses.append(valid_loss)

#     wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

#     scheduler.step(train_loss)

#     if epoch % 10 == 0:
#       print(f'Epoch: {epoch}')
#       print(f'\tTrain Loss: {train_loss:.4f}')
#       print(f'\t Val. Loss: {valid_loss:.4f}')
#     if valid_loss < best_valid_loss:
#       best_valid_loss = valid_loss

# plt.subplots(figsize=(5,3))
# plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
# plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
# plt.legend()
# plt.title(f'GNN Training and Validation loss',fontsize = 15)
# plt.xlabel('Epochs',fontsize = 12)
# plt.ylabel('MSE Loss',fontsize = 12)
# plt.semilogy()

# training_losses=np.array(training_losses)
# validation_losses=np.array(validation_losses)

In [ ]:
# torch.save(model.state_dict(), f"{system_size}_bus_N-1_GCN_5_256_PQVT.pth")
# wandb.save(f"{system_size}_bus_N-1_GCN_5_256_PQVT.pth")  # Upload to WandB

In [ ]:
test_loss, predictions, targets = test_model(model, test_loader)

In [ ]:
print('loss on test data is ', test_loss)

In [ ]:
predictions = torch.cat(predictions, dim=0)
targets = torch.cat(targets, dim=0)

In [ ]:
predictions = predictions.reshape(-1,system_size, 4)
targets = targets.reshape(-1, system_size, 4)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(predictions[:,:,0],targets[:,:,0])
voltage_magnitude_loss = calc_loss(predictions[:,:,1],targets[:,:,1])

In [ ]:
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(predictions[:,:,2],targets[:,:,2])
reactive_power_loss = calc_loss(predictions[:,:,3],targets[:,:,3])

In [ ]:
print('average active power discrepancy is  ', active_power_loss)
print('average reactive power discrepancy is  ', reactive_power_loss)

In [ ]:
# voltage_predictions = torch.cat((angle_predictions,v_predictions),2)
voltage_predictions = predictions[:,:,0:2]
voltage_targets = targets[:,:,0:2]

In [ ]:
def calculate_generator_power(
    demand: torch.Tensor, 
    voltage: torch.Tensor,  
    branches: list,  
    Yks: torch.Tensor,  
    Yij: torch.Tensor,  
    Yijc: torch.Tensor,  
    Tij: torch.Tensor,  
) -> torch.Tensor:
    num_nodes = voltage.shape[0]
    generator_power = torch.zeros_like(demand)
    
    # Helper function for complex multiplication
    def complex_mult(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[0] * b[0] - a[1] * b[1],
            a[0] * b[1] + a[1] * b[0]
        ])

    # Helper function for complex conjugate
    def complex_conj(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[0], -x[1]])

    # Helper function for complex division
    def complex_div(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[0]**2 + b[1]**2
        return torch.stack([
            (a[0] * b[0] + a[1] * b[1]) / denominator,
            (a[1] * b[0] - a[0] * b[1]) / denominator
        ])
    
    # Calculate shunt power terms for each node
    shunt_power = torch.zeros_like(demand)
    for i in range(num_nodes):
        v_mag_sq = voltage[i, 0]**2 + voltage[i, 1]**2
        v_mag_sq_tensor = torch.tensor([v_mag_sq, 0.0])
        shunt_power[i] = complex_mult(complex_conj(Yks[i]), v_mag_sq_tensor)
    # print('the shunt powers are: ', shunt_power)
    
    # Calculate branch flows
    branch_flows = {} 
    
    for idx, (i, j) in enumerate(branches):
        # Get complex voltage at both ends
        vi = voltage[i]
        vj = voltage[j]
        
        # First term of Sij
        vi_mag_sq = vi[0]**2 + vi[1]**2
        vi_mag_sq_tensor = torch.tensor([vi_mag_sq, 0.0], dtype=torch.float64)
        tij_mag_sq = Tij[idx, 0]**2 + Tij[idx, 1]**2
        tij_mag_sq_tensor = torch.tensor([tij_mag_sq, 0.0], dtype=torch.float64)
        vi_over_tij_sq = complex_div(vi_mag_sq_tensor, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance
        Y_total = torch.stack([
            Yij[idx, 0] + Yijc[idx, 0],
            Yij[idx, 1] + Yijc[idx, 1]
        ])
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        
        # Second term of Sij
        vivj = complex_mult(vi, complex_conj(vj))
        term2 = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vivj, Tij[idx])
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        # print(f'the branch flows for {i} , {j} in forward direction are: ', Sij)
        
        # Reverse flow Sji
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        vj_mag_sq = vj[0]**2 + vj[1]**2
        vj_mag_sq_tensor = torch.tensor([vj_mag_sq, 0.0], dtype=torch.float64)
        
        term1_ji = complex_mult(complex_conj(Y_total), vj_mag_sq_tensor)
        vjvi = complex_mult(complex_conj(vi), vj)
        term2_ji = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vjvi, complex_conj(Tij[idx]))
        )
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
        # print(f'the branch flows for {j} , {i} in reverse direction are: ', Sji)
    
    # Aggregate generator power for each node
    for i in range(num_nodes):
        
        for index, (from_bus, to_bus) in enumerate(branches): 
            if from_bus == i:
                generator_power[i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[i] += branch_flows[(to_bus, from_bus, index)]
        generator_power[i] += demand[i] + shunt_power[i]
    
    return generator_power, branch_flows


In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=1)

In [ ]:
def convert_to_complex_rectangle(tensor_2d):
    # Extract angle and magnitude
    tensor_mag = tensor_2d[:,0:1]  # In radians
    tensor_angle = tensor_2d[:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=1)

In [ ]:
test_x = torch.zeros((len(test_indices),test_dataset[0].x.shape[0],test_dataset[0].x.shape[1]))

for j in range(len(test_indices)):
    present_test_x = test_dataset[j].x
    test_x[j] = present_test_x

In [ ]:
load_input = test_x[:,:,15:17].cpu()

In [ ]:
voltage_predictions = voltage_predictions.cpu()
voltage_targets = voltage_targets.cpu()

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
Gen_Powers = []

Branch_Flows = []



In [ ]:
Yks = test_x[:,:,17:]
Yks = Yks[:,:, [1, 0]]

In [ ]:
for r in range(len(test_indices)):
    present_edge_input = test_dataset[r].edge_attr
    edge_g, edge_b = compute_gandb(present_edge_input)
    conductance_susceptance = torch.cat((edge_g, edge_b), dim=1)
    conductance_susceptance = conductance_susceptance.to('cpu')
    charging_susceptance = torch.zeros_like(conductance_susceptance)
    charging_susceptance[:,1:] =  present_edge_input[:,2:3].to('cpu')
    Tij = present_edge_input[:,9:].to('cpu')
    Tij_rec = convert_to_complex_rectangle(Tij)
    # print(Tij_rec.shape)
    complex_v = convert_to_complex_voltage(voltage_predictions[r])
    single_Yks = Yks[r].to('cpu')
    load= load_input[r].to('cpu')
    Branches = list(zip(*test_dataset[r].edge_index.cpu().tolist()))
    # print(len(Branches))
    gen_injection,branch_flows = calculate_generator_power(load,complex_v,Branches,single_Yks,conductance_susceptance,charging_susceptance,Tij_rec)
    Gen_Powers.append(gen_injection)
    Branch_Flows.append(branch_flows)

In [ ]:
generator_power_balance = torch.stack(Gen_Powers,dim=0)

In [ ]:
def compute_optimality(test_inputs, test_outputs, test_objective):

    test_inputs = test_inputs.cpu()
    test_outputs = test_outputs.cpu()
    test_objective = test_objective.cpu()

    c2 = test_inputs[:,:,12:13]
    c1 = test_inputs[:,:,13:14]
    c0 = test_inputs[:,:,14:15]

    # Get relevant output dimensions (zero-indexed)
    p_gens = test_outputs[:,:,2:3] # select on Pgs for generators
  

    print('the shape of c2 is ', c2.shape)
    print('the shape of p_gen is ', p_gens.shape)
    
    # Compute node-wise metrics
    system_metrics = c2 * (p_gens ** 2) + c1 * p_gens + c0


    model_obj = torch.sum(system_metrics, dim=(1,2))

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {test_objective.mean()}')

    print('the shape of test objective is ', test_objective.shape)
    print('the shape of model objective is ', model_obj.shape)

    optimality_gap = (model_obj / test_objective) * 100

    
    return optimality_gap.mean()

In [ ]:
test_objective = torch.zeros((len(test_indices)))

for j in range(len(test_indices)):
    present_test_obj = test_dataset[j].objective
    test_objective[j] = present_test_obj

In [ ]:
opt_gap = compute_optimality(test_x, predictions, test_objective)

In [ ]:
print('optimality gap now is ', opt_gap)

In [ ]:
voltage_targets = targets[:,:,0:2]

In [ ]:
predicted_angle_differences = []
predicted_angles = voltage_predictions[:,:,0]

for j in range(voltage_predictions.shape[0]):
    
    branch_for_this = list(zip(*test_dataset[j].edge_index.cpu().tolist()))
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    predicted_angle_differences.append(angle_differences)
    

In [ ]:
true_angle_differences = []
true_angles = voltage_targets[:,:,0]

for j in range(voltage_targets.shape[0]):
    
    branch_for_this = list(zip(*test_dataset[j].edge_index.cpu().tolist()))
    angle_differences = calculate_angle_differences(true_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    true_angle_differences.append(angle_differences)

In [ ]:
predicted_angle_differences = torch.cat(predicted_angle_differences)
true_angle_differences = torch.cat(true_angle_differences)

In [ ]:
# voltage angle difference bound 


angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max() )
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
## voltage magnitude bound

vmin = test_x[:,:,2:3].to('cpu')

vmax = test_x[:,:,3:4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - voltage_predictions[:,:,1:2], min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(voltage_predictions[:,:,1:2] - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max() )
print('average voltage magnitude violation is : ',vmag_violations.mean() )

In [ ]:
# Gen active power bounds 
pmin = test_x[:,:,6:7].to('cpu')

pmax = test_x[:,:,7:8].to('cpu')


p_gens = predictions[:,:,2:3]


lower_pgen_violations = torch.clamp(pmin - p_gens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(p_gens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max() )
print('average active power violation is : ',pgen_violations.mean() )

In [ ]:
# Gen reactive power bounds 
qmin = test_x[:,:,9:10].to('cpu')

qmax = test_x[:,:,10:11].to('cpu')


q_gens = predictions[:,:,3:4]


lower_qgen_violations = torch.clamp(qmin - q_gens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(q_gens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max() )
print('average reactive power violation is : ',qgen_violations.mean() )

In [ ]:
# compute the power magnitudes

# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 


    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
for_flow_vio = []
rev_flow_vio = []

In [ ]:
# Branch_Flows[0]

In [ ]:
for k in range(len(Branch_Flows)):
    flow_branch = list(zip(*test_dataset[k].edge_index.cpu().tolist()))
    
    # forward_keys = [(i, j, index) for index, (i,j) in enumerate(flow_branch)]
    # reverse_keys = [(j, i, index) for index, (i,j) in enumerate(flow_branch)]
    
    ##instead of doing the above, just select every other key from the branchflow dictionary, skip 1 for reverse
    
    all_keys = [key for key in Branch_Flows[k].keys()]
    forward_keys = all_keys[::2]
    reverse_keys = all_keys[1::2]
    
    forward_branch_flows = {key: Branch_Flows[k][key] for key in forward_keys if key in Branch_Flows[k]}
    reverse_branch_flows = {key: Branch_Flows[k][key] for key in reverse_keys if key in Branch_Flows[k]}


    forward_power_flows_list = [tensor for tensor in forward_branch_flows.values()]
    
    
    # Step 2: Concatenate tensors along the second axis (dim=1)
    forward_power_flows = torch.cat(forward_power_flows_list, dim=0).reshape(-1,2)



    reverse_power_flows = [tensor for tensor in reverse_branch_flows.values()]
    # Step 2: Concatenate tensors along the second axis (dim=1)
    reverse_power_flows = torch.cat(reverse_power_flows, dim=0).reshape(-1,2)
    forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
    reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)
    # Branch flow bounds in forward direction

    long_term_line_rating = test_dataset[k].edge_attr[:,6:7].to('cpu')
    branch_flow_limit = long_term_line_rating
    forward_branch_flow = forward_flow_magnitude
    forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound
    for_flow_vio.append(forward_flow_violations)
    reverse_branch_flow = reverse_flow_magnitude
    reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound
    rev_flow_vio.append(reverse_flow_violations)



In [ ]:
for_flow_vio = torch.cat(for_flow_vio)
rev_flow_vio = torch.cat(rev_flow_vio)

In [ ]:
print('max forward power flow violation is : ',for_flow_vio.max())
print('average  forward power flow violation is : ',for_flow_vio.mean())

In [ ]:
print('max reverse power flow violation is : ', rev_flow_vio.max())
print('average reverse power flow violation is : ', rev_flow_vio.mean())

In [ ]:
## Evaluate power balance constraint violations


real_power_balance_mismatches = generator_power_balance[:,:,0] - predictions[:,:,2]


print('max active power balance mismatch is : ', real_power_balance_mismatches.max() )
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean() )

In [ ]:
reactive_power_balance_mismatches = generator_power_balance[:,:,1] - predictions[:,:,3]


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max() )
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns=["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", for_flow_vio.max())
model_metrics_table.add_data("average forward power flows violation", for_flow_vio.mean())
model_metrics_table.add_data("max reverse power flows violation", rev_flow_vio.max())
model_metrics_table.add_data("average reverse power flows violation", rev_flow_vio.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()